# PathSnap — Notebook 01: Dataset Exploration (EDA)
**Milestone 2 · Owner: A (kumarsaavan07-jpg)**

**Partitions used:** `train` (80 trips) and `student_test` (10 trips). Each table/figure states the partition.
The 10 evaluator trips are **not** available to us and are never touched.

**What this notebook checks** (from *Dataset Exploration.pdf* §4):
trip counts · observations per trip · file availability & sizes · dtypes & columns · missing/invalid GPS ·
timestamp ordering & gaps · geographic extent · road-network size · connectivity (components, isolated nodes) ·
anomalies · reference-route structure.

Extra checks that matter for later stages: choosing a coordinate projection (§5), how far GPS points sit from the
reference route (label quality, §6), and how many road arcs lie near each point (candidate-set size for M3).

**Run on Colab:** upload `Student folder.zip` to Google Drive at `MyDrive/PathSnap/`, then *Runtime → Run all*.

In [ ]:
# ---- 1. Setup & configuration -------------------------------------------------------------
import os, sys, zipfile, time, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_PATH = "/content/drive/MyDrive/PathSnap/Student folder.zip"   # <-- change if your path differs
    EXTRACT_DIR = "/content/pathsnap_data"
    OUT_DIR = "/content/drive/MyDrive/PathSnap/eda_outputs"            # saved to Drive so it persists
else:  # local run (same code, paths from environment variables)
    ZIP_PATH = os.environ.get("PATHSNAP_ZIP", "Student folder.zip")
    EXTRACT_DIR = os.environ.get("PATHSNAP_EXTRACT", "pathsnap_data")
    OUT_DIR = os.environ.get("PATHSNAP_OUT", "eda_outputs")

if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)

# locate the folder that contains train/ and student_test/
DATA_ROOT = None
for dirpath, dirnames, _ in os.walk(EXTRACT_DIR):
    if "train" in dirnames and "student_test" in dirnames:
        DATA_ROOT = dirpath
        break
assert DATA_ROOT is not None, "Could not find train/ and student_test/ inside the extracted zip"
for sub in ["tables", "figures", "maps"]:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)

try:
    display
except NameError:          # outside Jupyter
    display = print
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("DATA_ROOT =", DATA_ROOT)
print("OUT_DIR   =", OUT_DIR)

## 2. Inventory: partitions, files, sizes, row counts
We do **not** assume every trip folder is identical. For every trip we check which of the four expected files
exist, their sizes, row counts, and the number of columns per row.

In [ ]:
PARTITIONS = ["train", "student_test"]
EXT = ["track", "nodes", "arcs", "route"]
EXPECTED_COLS = {"track": 3, "nodes": 2, "arcs": 2, "route": 1}

inv_rows = []
for part in PARTITIONS:
    for tid in sorted(os.listdir(os.path.join(DATA_ROOT, part))):
        tdir = os.path.join(DATA_ROOT, part, tid)
        if not os.path.isdir(tdir):
            continue
        row = {"partition": part, "trip_id": int(tid), "n_files": len(os.listdir(tdir))}
        for e in EXT:
            f = os.path.join(tdir, f"{tid}.{e}")
            row[f"has_{e}"] = os.path.isfile(f)
            if row[f"has_{e}"]:
                row[f"{e}_kb"] = round(os.path.getsize(f) / 1024, 1)
                with open(f) as fh:
                    ncols = [len(line.split()) for line in fh if line.strip()]
                row[f"{e}_rows"] = len(ncols)
                row[f"{e}_cols_ok"] = all(c == EXPECTED_COLS[e] for c in ncols)
        inv_rows.append(row)
inventory = pd.DataFrame(inv_rows)
inventory.to_csv(os.path.join(OUT_DIR, "tables", "01_inventory.csv"), index=False)

print("Trips per partition:\n", inventory.groupby("partition").size(), "\n")
print("All 4 files present in every trip:", bool(inventory[[f"has_{e}" for e in EXT]].all().all()))
print("Column counts as expected in every file:", bool(inventory[[f"{e}_cols_ok" for e in EXT]].all().all()))
print("Trip-ID overlap between partitions:",
      set(inventory.query("partition=='train'").trip_id) & set(inventory.query("partition=='student_test'").trip_id))
display(inventory.groupby("partition")[["track_rows", "nodes_rows", "arcs_rows", "route_rows"]]
        .describe().T.round(1))

## 3. Loading trips + data types
Columns have no headers; we name them from the dataset description and **check** the inferred pandas dtypes.

In [ ]:
def read_trip(part, tid):
    """Read one trip's four files. Returns dict of DataFrames (no processing applied)."""
    base = os.path.join(DATA_ROOT, part, f"{tid:08d}", f"{tid:08d}")
    rd = lambda e, names: pd.read_csv(f"{base}.{e}", sep=r"\s+", header=None, names=names)
    return {
        "track": rd("track", ["lon", "lat", "t"]),
        "nodes": rd("nodes", ["lon", "lat"]),
        "arcs":  rd("arcs", ["from_node", "to_node"]),
        "route": rd("route", ["arc_id"]),
    }

t0 = time.time()
TRIPS = {(r.partition, r.trip_id): read_trip(r.partition, r.trip_id) for r in inventory.itertuples()}
print(f"Loaded {len(TRIPS)} trips in {time.time()-t0:.1f}s")

dtype_rows = []
for (part, tid), d in TRIPS.items():
    for name, df in d.items():
        dtype_rows.append({"file": name, **{c: str(t) for c, t in df.dtypes.items()}})
dtypes = pd.DataFrame(dtype_rows).groupby("file").agg(lambda s: ",".join(sorted(set(s.dropna()))))
display(dtypes)

## 4. Coordinate system — choosing a metric projection
Longitude/latitude are angles, so Euclidean distance in degrees is **not** a distance. Trips are spread across the
world (both hemispheres, both sides of Greenwich), so one global projection won't do.

**Choice:** for each trip, a local *equirectangular* projection centred on the trip's mean position:
`x = R·(lon−lon0)·cos(lat0)`, `y = R·(lat−lat0)` in metres, R = 6 371 008.8 m (mean Earth radius).
It is cheap, has no extra dependencies, and is very accurate over a few tens of km. **We check this** below
against the exact great-circle (haversine) distance, both for consecutive GPS points (the local scale that
matters for point-to-road distances) and for random far-apart pairs (worst case).

In [ ]:
R_EARTH = 6371008.8

def haversine(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, (lon1, lat1, lon2, lat2))
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

def make_projection(lon0, lat0):
    c = math.cos(math.radians(lat0))
    def to_xy(lon, lat):
        lon, lat = np.asarray(lon, float), np.asarray(lat, float)
        return np.column_stack([np.radians(lon - lon0) * R_EARTH * c, np.radians(lat - lat0) * R_EARTH])
    return to_xy

proj_rows = []
for (part, tid), d in TRIPS.items():
    tr = d["track"]
    to_xy = make_projection(tr.lon.mean(), tr.lat.mean())
    xy = to_xy(tr.lon, tr.lat)
    # consecutive pairs
    h = haversine(tr.lon.values[:-1], tr.lat.values[:-1], tr.lon.values[1:], tr.lat.values[1:])
    e = np.linalg.norm(np.diff(xy, axis=0), axis=1)
    m = h > 1.0
    rel_local = np.abs(e[m] - h[m]) / h[m]
    # random far pairs
    i, j = rng.integers(0, len(tr), 2000), rng.integers(0, len(tr), 2000)
    hf = haversine(tr.lon.values[i], tr.lat.values[i], tr.lon.values[j], tr.lat.values[j])
    ef = np.linalg.norm(xy[i] - xy[j], axis=1)
    mf = hf > 100
    rel_far = np.abs(ef[mf] - hf[mf]) / hf[mf]
    proj_rows.append({"partition": part, "trip_id": tid,
                      "lat0": tr.lat.mean(),
                      "consec_p99_rel_err_%": 100 * np.percentile(rel_local, 99) if m.any() else np.nan,
                      "consec_max_abs_err_m": np.max(np.abs(e[m] - h[m])) if m.any() else np.nan,
                      "far_max_rel_err_%": 100 * rel_far.max() if mf.any() else np.nan})
proj_check = pd.DataFrame(proj_rows)
proj_check.to_csv(os.path.join(OUT_DIR, "tables", "02_projection_check.csv"), index=False)
display(proj_check.describe().T.round(4))

## 5. GPS trajectories (.track): missing/invalid values, timestamps, gaps, extent, speed

In [ ]:
LARGE_GAP_S = 10        # a jump of >10 s between fixes we call a "gap" (nominal rate is 1 Hz)
SPEED_JUMP_MS = 60      # >60 m/s (216 km/h) between fixes = implausible jump for a road vehicle

gps_rows, all_dt, all_speed = [], [], []
for (part, tid), d in TRIPS.items():
    tr = d["track"]
    n = len(tr)
    n_nan = int(tr.isna().any(axis=1).sum())
    n_bad_range = int(((tr.lon.abs() > 180) | (tr.lat.abs() > 90)).sum())
    dt = np.diff(tr.t.values)
    xy = make_projection(tr.lon.mean(), tr.lat.mean())(tr.lon, tr.lat)
    step = np.linalg.norm(np.diff(xy, axis=0), axis=1)
    speed = step / np.where(dt > 0, dt, np.nan)
    # dwell ("parked"): net displacement over a 60-sample window, divided by elapsed time, < 0.5 m/s.
    # Short windows fail because parked GPS (esp. indoors) jitters over 100+ m; a 60-s window averages that out.
    w = 60
    v60 = np.full(n, np.nan)
    if n > w:
        v60[:-w] = np.linalg.norm(xy[w:] - xy[:-w], axis=1) / np.maximum(tr.t.values[w:] - tr.t.values[:-w], 1)
    path10_km = np.linalg.norm(np.diff(xy[::10], axis=0), axis=1).sum() / 1000
    all_dt.append(pd.DataFrame({"partition": part, "dt": dt}))
    all_speed.append(pd.DataFrame({"partition": part, "speed": speed[dt <= 1]}))   # 1-s steps only
    gps_rows.append({
        "partition": part, "trip_id": tid, "n_points": n,
        "n_missing_rows": n_nan, "n_out_of_range": n_bad_range,
        "n_dup_timestamps": int((dt == 0).sum()), "time_monotonic": bool((dt > 0).all()),
        "t_start": tr.t.iloc[0], "duration_min": (tr.t.iloc[-1] - tr.t.iloc[0]) / 60,
        "frac_dt_eq_1": float((dt == 1).mean()), "n_gaps_gt10s": int((dt > LARGE_GAP_S).sum()),
        "n_gaps_gt60s": int((dt > 60).sum()), "max_gap_s": float(dt.max()),
        "path_km": step.sum() / 1000, "path_km_10s_subsampled": path10_km,
        "extent_ew_km": np.ptp(xy[:, 0]) / 1000, "extent_ns_km": np.ptp(xy[:, 1]) / 1000,
        "speed_med_ms": float(np.nanmedian(speed)), "speed_p95_ms": float(np.nanpercentile(speed, 95)),
        "speed_p99_ms": float(np.nanpercentile(speed, 99)),
        "n_speed_jumps": int(np.nansum(speed > SPEED_JUMP_MS)),
        "frac_dwell_60s": float(np.nanmean(v60[:-w] < 0.5)) if n > w else np.nan,
        "lon0": tr.lon.mean(), "lat0": tr.lat.mean(),
    })
gps = pd.DataFrame(gps_rows)
all_dt = pd.concat(all_dt, ignore_index=True)
all_speed = pd.concat(all_speed, ignore_index=True)
gps.to_csv(os.path.join(OUT_DIR, "tables", "03_gps_stats.csv"), index=False)

print("Missing rows total:", gps.n_missing_rows.sum(), "| out-of-range coords:", gps.n_out_of_range.sum(),
      "| duplicate timestamps:", gps.n_dup_timestamps.sum(),
      "| all timestamps strictly increasing:", gps.time_monotonic.all())
print("Share of ALL time steps that are exactly 1 s:", round(float((all_dt.dt == 1).mean()), 4))
display(gps.groupby("partition")[["n_points", "duration_min", "path_km", "frac_dt_eq_1", "max_gap_s",
                                   "speed_med_ms", "speed_p99_ms", "n_speed_jumps", "frac_dwell_60s"]].describe().T.round(2))
print("\nTrips with the largest time gaps:")
display(gps.sort_values("max_gap_s", ascending=False).head(10)[
    ["partition", "trip_id", "n_points", "max_gap_s", "n_gaps_gt10s", "n_gaps_gt60s", "frac_dt_eq_1"]])

## 6. Road network (.nodes + .arcs): size, validity, connectivity
Arcs are **directed** (`from_node → to_node`). A two-way street appears as two arcs (u→v and v→u).
Connectivity is measured with weakly connected components (ignore direction) and strongly connected
components (respect direction: can you drive from every node to every other?).

In [ ]:
def arc_lengths_m(nodes_xy, arcs):
    return np.linalg.norm(nodes_xy[arcs[:, 1]] - nodes_xy[arcs[:, 0]], axis=1)

NET = {}   # cache projected network per trip for later sections
net_rows = []
for (part, tid), d in TRIPS.items():
    tr, nd, ar = d["track"], d["nodes"], d["arcs"].values
    to_xy = make_projection(tr.lon.mean(), tr.lat.mean())
    nxy = to_xy(nd.lon, nd.lat)
    n_nodes, n_arcs = len(nd), len(ar)
    bad_ref = int(((ar < 0) | (ar >= n_nodes)).any(axis=1).sum())
    self_loops = int((ar[:, 0] == ar[:, 1]).sum())
    dup_arcs = n_arcs - len(np.unique(ar, axis=0))
    used = np.zeros(n_nodes, bool); used[ar.ravel()] = True
    key = ar[:, 0].astype(np.int64) * n_nodes + ar[:, 1]
    rkey = ar[:, 1].astype(np.int64) * n_nodes + ar[:, 0]
    two_way = np.isin(rkey, key) & (ar[:, 0] != ar[:, 1])
    G = coo_matrix((np.ones(n_arcs), (ar[:, 0], ar[:, 1])), shape=(n_nodes, n_nodes)).tocsr()
    n_wcc, wcc_lab = connected_components(G, directed=True, connection="weak")
    n_scc, scc_lab = connected_components(G, directed=True, connection="strong")
    big_scc = np.bincount(scc_lab).argmax()
    L = arc_lengths_m(nxy, ar)
    NET[(part, tid)] = {"nxy": nxy, "arcs": ar, "len": L, "scc_lab": scc_lab, "big_scc": big_scc, "to_xy": to_xy}
    net_rows.append({
        "partition": part, "trip_id": tid, "n_nodes": n_nodes, "n_arcs": n_arcs,
        "arcs_bad_node_ref": bad_ref, "self_loops": self_loops, "duplicate_arcs": dup_arcs,
        "isolated_nodes": int((~used).sum()), "frac_two_way_arcs": float(two_way.mean()),
        "arc_len_med_m": float(np.median(L)), "arc_len_p95_m": float(np.percentile(L, 95)),
        "arc_len_max_m": float(L.max()), "zero_len_arcs": int((L < 0.01).sum()),
        "n_weak_components": n_wcc, "largest_wcc_frac": float(np.bincount(wcc_lab).max() / n_nodes),
        "n_strong_components": n_scc, "largest_scc_frac": float(np.bincount(scc_lab).max() / n_nodes),
        "net_extent_ew_km": np.ptp(nxy[:, 0]) / 1000, "net_extent_ns_km": np.ptp(nxy[:, 1]) / 1000,
    })
net = pd.DataFrame(net_rows)
net.to_csv(os.path.join(OUT_DIR, "tables", "04_network_stats.csv"), index=False)
display(net.groupby("partition")[["n_nodes", "n_arcs", "self_loops", "duplicate_arcs", "isolated_nodes",
                                   "frac_two_way_arcs", "arc_len_med_m", "arc_len_max_m",
                                   "n_weak_components", "largest_wcc_frac", "largest_scc_frac"]]
        .describe().T.round(3))

## 7. Reference route (.route): encoding, validity, continuity
`.route` lists the arcs the vehicle drove, **in order**. A valid continuous route needs
`to_node(arc_k) == from_node(arc_{k+1})`. We count where that fails ("breaks"), repeated arcs, how the route's
length compares with the GPS path length, and whether route arcs sit in the network's largest strongly-connected part.

In [ ]:
route_rows = []
for (part, tid), d in TRIPS.items():
    rt = d["route"].arc_id.values
    N = NET[(part, tid)]; ar = N["arcs"]
    valid = (rt >= 0) & (rt < len(ar))
    rtv = rt[valid]
    connected = ar[rtv[:-1], 1] == ar[rtv[1:], 0]
    same = rtv[:-1] == rtv[1:]
    route_len_km = N["len"][rtv].sum() / 1000
    in_big = N["scc_lab"][ar[rtv, 0]] == N["big_scc"]
    route_rows.append({
        "partition": part, "trip_id": tid, "n_route_arcs": len(rt), "n_invalid_ids": int((~valid).sum()),
        "n_breaks": int((~connected & ~same).sum()), "continuity": float(connected.mean()),
        "n_consecutive_repeats": int(same.sum()), "n_unique_arcs": len(np.unique(rtv)),
        "route_km": route_len_km,
        "route_to_gps_len_ratio": route_len_km / gps.set_index(["partition", "trip_id"]).loc[(part, tid), "path_km"],
        "frac_route_in_largest_scc": float(in_big.mean()),
        "pts_per_route_arc": len(d["track"]) / len(rt),
    })
routes = pd.DataFrame(route_rows)
routes.to_csv(os.path.join(OUT_DIR, "tables", "05_route_stats.csv"), index=False)
display(routes.groupby("partition")[["n_route_arcs", "n_invalid_ids", "n_breaks", "continuity",
                                      "n_unique_arcs", "route_km", "route_to_gps_len_ratio",
                                      "frac_route_in_largest_scc", "pts_per_route_arc"]].describe().T.round(3))
print("Trips whose reference route has breaks:")
display(routes[routes.n_breaks > 0][["partition", "trip_id", "n_route_arcs", "n_breaks", "continuity"]])

## 8. GPS ↔ reference route geometry (label quality)
For every GPS point: distance (m) to the nearest segment of its own reference route. This tells us
(a) how noisy GPS is relative to the road, and (b) where the reference route and the GPS disagree —
e.g. the route starts late / ends early, or points are far from any route arc. This is evidence for §6 of the
dataset guide ("reference routes may have limitations").

In [ ]:
def point_to_segments(P, A, B, chunk=1500):
    """Exact distance from each point in P (n,2) to each segment A[k]→B[k] (m,2). Returns (n,m) distances."""
    AB = B - A
    L2 = np.maximum((AB ** 2).sum(1), 1e-12)
    out = np.empty((len(P), len(A)))
    for s in range(0, len(P), chunk):
        AP = P[s:s + chunk, None, :] - A[None]
        t = np.clip((AP * AB[None]).sum(2) / L2[None], 0, 1)
        out[s:s + chunk] = np.linalg.norm(AP - t[..., None] * AB[None], axis=2)
    return out

geo_rows, all_d2route = [], []
for (part, tid), d in TRIPS.items():
    N = NET[(part, tid)]; ar = N["arcs"]
    rt = d["route"].arc_id.values
    rt = rt[(rt >= 0) & (rt < len(ar))]
    P = N["to_xy"](d["track"].lon, d["track"].lat)
    D = point_to_segments(P, N["nxy"][ar[rt, 0]], N["nxy"][ar[rt, 1]])
    dmin = D.min(1)
    N["d2route"] = dmin
    all_d2route.append(pd.DataFrame({"partition": part, "trip_id": tid, "d2route": dmin}))
    geo_rows.append({"partition": part, "trip_id": tid,
                     "d2route_med_m": np.median(dmin), "d2route_p90_m": np.percentile(dmin, 90),
                     "d2route_p99_m": np.percentile(dmin, 99), "d2route_max_m": dmin.max(),
                     "frac_gt20m": float((dmin > 20).mean()), "frac_gt50m": float((dmin > 50).mean()),
                     "first_pt_d2route_m": dmin[0], "last_pt_d2route_m": dmin[-1]})
geo = pd.DataFrame(geo_rows)
all_d2route = pd.concat(all_d2route, ignore_index=True)
geo.to_csv(os.path.join(OUT_DIR, "tables", "06_gps_to_route.csv"), index=False)
print("All points, distance to own reference route (m):")
display(all_d2route.groupby("partition").d2route.describe(percentiles=[.5, .9, .95, .99]).round(2))
print("Trips with the most points >50 m from their route:")
display(geo.sort_values("frac_gt50m", ascending=False).head(8).round(3))

## 9. GPS ↔ whole road network: nearest arc and a first look at candidate-set size
How far is each GPS point from the **nearest arc of any kind**, and roughly how many arcs lie within 25 / 50 / 100 m?
Also a naive baseline: *is the nearest arc on the reference route?* (ignoring direction and order).

⚠️ **Preliminary only.** The 25/50/100 m counts here use a random 1,000 points per trip (so every trip weighs the
same) and ignore driving direction. The candidate rule is chosen **precisely in section 10** (all points,
heading-aware reference arc, train-only selection).

Method: arcs near the trajectory are densified into points every 10 m and put into a KD-tree;
the nearest few hits are then refined with the exact point-to-segment distance.

In [ ]:
DENSIFY_STEP = 10.0
RADII = [25, 50, 100]
SUB_N = 1000      # random subsample of points per trip for the (slower) candidate-count analysis

def densify(nxy, arcs, idx, step=DENSIFY_STEP):
    A, B = nxy[arcs[idx, 0]], nxy[arcs[idx, 1]]
    L = np.linalg.norm(B - A, axis=1)
    k = np.maximum(1, np.ceil(L / step).astype(int))
    rep = np.repeat(np.arange(len(idx)), k + 1)
    t = np.concatenate([np.linspace(0, 1, kk + 1) for kk in k])
    pts = A[rep] + t[:, None] * (B - A)[rep]
    return pts, idx[rep]

cand_rows, nearest_rows = [], []
for (part, tid), d in TRIPS.items():
    N = NET[(part, tid)]; ar, nxy = N["arcs"], N["nxy"]
    P = N["to_xy"](d["track"].lon, d["track"].lat)
    lo, hi = P.min(0) - 300, P.max(0) + 300
    A, B = nxy[ar[:, 0]], nxy[ar[:, 1]]
    inbox = np.where((np.maximum(A, B) >= lo).all(1) & (np.minimum(A, B) <= hi).all(1))[0]
    dpts, darc = densify(nxy, ar, inbox)
    tree = cKDTree(dpts)
    _, nn = tree.query(P, k=16)
    cand = darc[nn]                                      # (n,16) arc ids
    dd = np.full(cand.shape, np.inf)
    for j in range(cand.shape[1]):
        a = cand[:, j]
        AB = nxy[ar[a, 1]] - nxy[ar[a, 0]]
        AP = P - nxy[ar[a, 0]]
        t = np.clip((AP * AB).sum(1) / np.maximum((AB ** 2).sum(1), 1e-12), 0, 1)
        dd[:, j] = np.linalg.norm(AP - t[:, None] * AB, axis=1)
    best = cand[np.arange(len(P)), dd.argmin(1)]
    dnear = dd.min(1)
    rt = set(d["route"].arc_id.values.tolist())
    rt_und = {(min(ar[a, 0], ar[a, 1]), max(ar[a, 0], ar[a, 1])) for a in rt if 0 <= a < len(ar)}
    on_dir = np.array([b in rt for b in best])
    on_und = np.array([(min(ar[b, 0], ar[b, 1]), max(ar[b, 0], ar[b, 1])) in rt_und for b in best])
    nearest_rows.append({"partition": part, "trip_id": tid,
                         "nearest_arc_med_m": np.median(dnear), "nearest_arc_p95_m": np.percentile(dnear, 95),
                         "nearest_arc_max_m": dnear.max(), "frac_off_network_gt50m": float((dnear > 50).mean()),
                         "frac_nearest_on_route_directed": on_dir.mean(),
                         "frac_nearest_on_route_undirected": on_und.mean(),
                         "frac_d2route_minus_nearest_gt5m": float(((N["d2route"] - dnear) > 5).mean())})
    # candidate counts on a subsample
    sub = rng.choice(len(P), min(SUB_N, len(P)), replace=False)
    for r in RADII:
        hits = tree.query_ball_point(P[sub], r=r + DENSIFY_STEP / 2)
        for i, h in zip(sub, hits):
            arcs_h = np.unique(darc[h]) if len(h) else np.array([], int)
            if len(arcs_h):
                AB = nxy[ar[arcs_h, 1]] - nxy[ar[arcs_h, 0]]
                AP = P[i] - nxy[ar[arcs_h, 0]]
                t = np.clip((AP * AB).sum(1) / np.maximum((AB ** 2).sum(1), 1e-12), 0, 1)
                arcs_h = arcs_h[np.linalg.norm(AP - t[:, None] * AB, axis=1) <= r]
            cand_rows.append({"partition": part, "trip_id": tid, "radius_m": r, "n_candidates": len(arcs_h),
                              "route_arc_within_r": bool(len(set(arcs_h.tolist()) & rt))})
nearest = pd.DataFrame(nearest_rows)
cands = pd.DataFrame(cand_rows)
nearest.to_csv(os.path.join(OUT_DIR, "tables", "07_nearest_arc.csv"), index=False)
cands.groupby(["partition", "radius_m"]).agg(
    mean_candidates=("n_candidates", "mean"), median_candidates=("n_candidates", "median"),
    p95_candidates=("n_candidates", lambda s: s.quantile(.95)),
    frac_any_route_arc_within_r=("route_arc_within_r", "mean")).round(3).to_csv(
    os.path.join(OUT_DIR, "tables", "08_candidate_counts_by_radius.csv"))
display(nearest.groupby("partition").describe().T.round(3).loc[
    lambda x: x.index.get_level_values(1).isin(["mean", "50%", "min", "max"])])
display(pd.read_csv(os.path.join(OUT_DIR, "tables", "08_candidate_counts_by_radius.csv")))

## 10. Choosing the candidate rule precisely (train trips select, student-test only checks)
For each GPS point we compute:
* **d\*** = distance to its reference arc: the nearest route arc pointing the **same way** the vehicle moves
  (heading from positions 4 s apart, only when speed > 3 m/s); if not moving, the nearest route arc.
* **d_nearest** = distance to the nearest candidate arc (zero-length arcs dropped, duplicate (u,v) arcs merged — D11).
* the number of candidate arcs **K** under two rules:
  * **fixed radius r**: all arcs within r metres (5–150 m);
  * **adaptive**: all arcs within min(d_nearest + δ, 150 m) — the window is measured from the closest road, so it
    adapts to local GPS error.

**Coverage** = share of points whose reference arc is inside the candidate set = 1 − candidate miss rate.
Cost = mean K (classifier rows, class imbalance) and mean K² (Viterbi work per step).
Trips 54, 33, 2 (parked off-network / route ends early, flagged in sections 5–8 independently of any radius) are
reported separately: no radius can fix them, and they would otherwise dominate the averages.

In [ ]:
RC_STEP = 5.0                    # densify step (m) for the KD-tree
RC_RMAX = 150.0                  # hard cap (m)
RC_RADII = list(range(5, 151, 5))
RC_DELTAS = [5, 10, 15, 20, 25, 30, 40, 50]
ANOM_TRIPS = [54, 33, 2]

def bearing_deg(v):
    return np.degrees(np.arctan2(v[:, 0], v[:, 1])) % 360       # 0 = north, clockwise

def seg_dist(P, A, B):
    AB = B - A; AP = P - A
    t = np.clip((AP * AB).sum(-1) / np.maximum((AB ** 2).sum(-1), 1e-12), 0, 1)
    return np.linalg.norm(AP - t[..., None] * AB, axis=-1)

rc_rows = []
t0 = time.time()
for (part, tid), d in TRIPS.items():
    N = NET[(part, tid)]; ar, nxy = N["arcs"], N["nxy"]
    tr = d["track"]; P = N["to_xy"](tr.lon, tr.lat)
    # candidate arcs: drop zero-length, merge exact duplicate (u, v) pairs
    keep = np.where(N["len"] >= 0.01)[0]
    _, first = np.unique(ar[keep], axis=0, return_index=True); keep = keep[np.sort(first)]
    # GPS heading where moving
    h = np.full(len(P), np.nan)
    if len(P) > 4:
        v = P[4:] - P[:-4]; sp = np.linalg.norm(v, axis=1) / np.maximum(tr.t.values[4:] - tr.t.values[:-4], 1)
        hh = bearing_deg(v); hh[sp < 3] = np.nan; h[2:-2] = hh
    # d* : heading-aware distance to the reference route
    rt = d["route"].arc_id.values
    A, B = nxy[ar[rt, 0]], nxy[ar[rt, 1]]; ab = bearing_deg(B - A)
    dstar = np.empty(len(P))
    for s0 in range(0, len(P), 1500):
        sl = slice(s0, s0 + 1500)
        D = seg_dist(P[sl, None], A[None], B[None])
        diff = np.abs((h[sl, None] - ab[None] + 180) % 360 - 180)
        dstar[sl] = np.where(np.isnan(h[sl, None]) | (diff < 90), D, np.inf).min(1)
    # every candidate arc within RC_RMAX: densify → KD-tree ball query → exact distance
    lo, hi = P.min(0) - RC_RMAX - 50, P.max(0) + RC_RMAX + 50
    Ak, Bk = nxy[ar[keep, 0]], nxy[ar[keep, 1]]
    inbox = np.where((np.maximum(Ak, Bk) >= lo).all(1) & (np.minimum(Ak, Bk) <= hi).all(1))[0]
    Lk = np.linalg.norm(Bk[inbox] - Ak[inbox], axis=1); kk = np.maximum(1, np.ceil(Lk / RC_STEP).astype(int))
    rep = np.repeat(np.arange(len(inbox)), kk + 1); tt = np.concatenate([np.linspace(0, 1, x + 1) for x in kk])
    dp = Ak[inbox][rep] + tt[:, None] * (Bk[inbox] - Ak[inbox])[rep]; darc = inbox[rep]
    tree = cKDTree(dp)
    K = {r: np.zeros(len(P), int) for r in RC_RADII}; KA = {dl: np.zeros(len(P), int) for dl in RC_DELTAS}
    dnear = np.full(len(P), np.inf)
    for s0 in range(0, len(P), 2000):
        n_ch = min(2000, len(P) - s0)
        hits = tree.query_ball_point(P[s0:s0 + n_ch], r=RC_RMAX + RC_STEP / 2)
        pi = np.repeat(np.arange(n_ch), [len(x) for x in hits])
        ai = darc[np.concatenate([np.asarray(x, int) for x in hits])] if len(pi) else np.array([], int)
        key = np.unique(pi.astype(np.int64) * 10_000_000 + ai); pi, ai = key // 10_000_000, key % 10_000_000
        dd = seg_dist(P[s0 + pi], Ak[ai], Bk[ai]); m = dd <= RC_RMAX; pi, dd = pi[m], dd[m]
        for r in RC_RADII:
            K[r][s0:s0 + n_ch] = np.bincount(pi[dd <= r], minlength=n_ch)
        mn = np.full(n_ch, np.inf); np.minimum.at(mn, pi, dd); dnear[s0:s0 + n_ch] = mn
        for dl in RC_DELTAS:
            KA[dl][s0:s0 + n_ch] = np.bincount(pi[dd <= np.minimum(mn[pi] + dl, RC_RMAX)], minlength=n_ch)
    rc_rows.append(pd.DataFrame({"partition": part, "trip_id": tid, "dstar": dstar, "dnear": dnear, "moving": ~np.isnan(h),
                                 **{f"K{r}": K[r] for r in RC_RADII}, **{f"KA{dl}": KA[dl] for dl in RC_DELTAS}}))
RC = pd.concat(rc_rows, ignore_index=True)
print(f"candidate analysis: {len(RC):,} points in {time.time() - t0:.0f}s")

rc_train = RC[RC.partition == "train"]; rc_clean = rc_train[~rc_train.trip_id.isin(ANOM_TRIPS)]
rc_test = RC[RC.partition == "student_test"]
mv = rc_train[rc_train.moving & np.isfinite(rc_train.dstar)]
sigma = 1.4826 * np.median(mv.dstar)       # robust GPS-noise estimate (Newson & Krumm 2009)
print(f"Robust GPS noise σ (train, moving) = {sigma:.2f} m | empirical d* p95 {mv.dstar.quantile(.95):.1f} m, "
      f"p99 {mv.dstar.quantile(.99):.1f} m | half-normal model would predict p95 {1.96*sigma:.1f} m, p99 {2.576*sigma:.1f} m")
print(f"Share of train points in anomalous trips {ANOM_TRIPS}: {rc_train.trip_id.isin(ANOM_TRIPS).mean():.3f}")

cov_f = lambda g, r: (g.dstar <= r).mean()
cov_a = lambda g, dl: (g.dstar <= np.minimum(g.dnear + dl, RC_RMAX)).mean()
fixed_tab = pd.DataFrame([{"radius_m": r, "coverage_train_all": cov_f(rc_train, r), "coverage_train_clean": cov_f(rc_clean, r),
                           "coverage_student_test": cov_f(rc_test, r), "mean_K_clean": rc_clean[f"K{r}"].mean(),
                           "mean_K2_clean": (rc_clean[f"K{r}"] ** 2).mean(), "empty_sets_clean": (rc_clean[f"K{r}"] == 0).mean()}
                          for r in RC_RADII])
fixed_tab["gain_next_5m_pp"] = (fixed_tab.coverage_train_clean.shift(-1) - fixed_tab.coverage_train_clean) * 100
adapt_tab = pd.DataFrame([{"delta_m": dl, "coverage_train_all": cov_a(rc_train, dl), "coverage_train_clean": cov_a(rc_clean, dl),
                           "coverage_student_test": cov_a(rc_test, dl), "mean_K_clean": rc_clean[f"KA{dl}"].mean(),
                           "mean_K2_clean": (rc_clean[f"KA{dl}"] ** 2).mean(), "empty_sets_clean": (rc_clean[f"KA{dl}"] == 0).mean()}
                          for dl in RC_DELTAS])
fixed_tab.to_csv(os.path.join(OUT_DIR, "tables", "11_candidate_rule_fixed_radius.csv"), index=False)
adapt_tab.to_csv(os.path.join(OUT_DIR, "tables", "12_candidate_rule_adaptive.csv"), index=False)
print("\nFIXED radius:"); display(fixed_tab[fixed_tab.radius_m.isin([10, 15, 20, 25, 30, 40, 50, 60, 80, 100])].round(4))
print("\nADAPTIVE d_nearest + δ (cap 150 m):"); display(adapt_tab.round(4))

# selection rule (train-clean only): smallest δ after which +5 m adds < 0.25 percentage points of coverage
ad = adapt_tab.set_index("delta_m").coverage_train_clean
gains = {dl: (ad[nx] - ad[dl]) * 100 / ((nx - dl) / 5) for dl, nx in zip(RC_DELTAS[:-1], RC_DELTAS[1:])}
CHOSEN_DELTA = min(dl for dl, g in gains.items() if g < 0.25)
RC_CHOSEN = adapt_tab.set_index("delta_m").loc[CHOSEN_DELTA]
r_eq = min(RC_RADII, key=lambda r: abs(rc_clean[f"K{r}"].mean() - RC_CHOSEN.mean_K_clean))
print(f"\nCHOSEN rule: candidates = arcs within min(d_nearest + {CHOSEN_DELTA} m, {RC_RMAX:.0f} m)")
print(f"  coverage: train-clean {RC_CHOSEN.coverage_train_clean:.4f} | train-all {RC_CHOSEN.coverage_train_all:.4f} | student-test (check) {RC_CHOSEN.coverage_student_test:.4f}")
print(f"  cost: mean K {RC_CHOSEN.mean_K_clean:.2f}, mean K² {RC_CHOSEN.mean_K2_clean:.1f}, empty candidate sets {RC_CHOSEN.empty_sets_clean:.4f}")
print(f"  same budget fixed radius r={r_eq} m: coverage {cov_f(rc_clean, r_eq):.4f} (K {rc_clean[f'K{r_eq}'].mean():.2f})")
print(f"  fixed 50 m for reference: coverage {cov_f(rc_clean, 50):.4f}, K {rc_clean['K50'].mean():.2f}, K² {(rc_clean['K50']**2).mean():.1f}, empty {(rc_clean['K50']==0).mean():.4f}")
mm = rc_clean[rc_clean.dstar > np.minimum(rc_clean.dnear + CHOSEN_DELTA, RC_RMAX)]
print(f"  remaining clean misses: {len(mm)} pts; their reference arc is a median {mm.dstar.median():.0f} m away "
      f"({mm.moving.mean():.0%} moving) → reference errors, not a radius problem")

# figure
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(fixed_tab.mean_K_clean, fixed_tab.coverage_train_clean, "o-", ms=4, label="fixed radius r", color="#1f77b4")
ax[0].plot(adapt_tab.mean_K_clean, adapt_tab.coverage_train_clean, "s-", ms=5, label="adaptive: d_nearest + δ", color="#d62728")
for _, rr in fixed_tab[fixed_tab.radius_m.isin([15, 25, 50, 100])].iterrows():
    ax[0].annotate(f"r={rr.radius_m:.0f}", (rr.mean_K_clean, rr.coverage_train_clean), textcoords="offset points", xytext=(5, -12), fontsize=8, color="#1f77b4")
for _, rr in adapt_tab[adapt_tab.delta_m.isin([10, 20, 30, 50])].iterrows():
    ax[0].annotate(f"δ={rr.delta_m:.0f}", (rr.mean_K_clean, rr.coverage_train_clean), textcoords="offset points", xytext=(-10, 6), fontsize=8, color="#d62728")
ax[0].scatter([RC_CHOSEN.mean_K_clean], [RC_CHOSEN.coverage_train_clean], s=160, facecolors="none", edgecolors="k", lw=1.5, zorder=5)
ax[0].set(xlim=(0, 25), ylim=(0.80, 1.0), xlabel="mean candidates per GPS point (cost)", ylabel="coverage = 1 − candidate miss rate",
          title=f"Coverage vs cost — train (excl. {ANOM_TRIPS})"); ax[0].grid(alpha=.3); ax[0].legend(loc="lower right")
ax[1].plot(adapt_tab.delta_m, adapt_tab.coverage_train_clean, "s-", label="train (clean) — selects")
ax[1].plot(adapt_tab.delta_m, adapt_tab.coverage_student_test, "^--", label="student-test — check only")
ax[1].axvline(CHOSEN_DELTA, ls=":", c="k")
ax[1].set(xlabel="δ (m) beyond the nearest arc", ylabel="coverage", ylim=(0.9, 1.0), title=f"Adaptive rule (chosen δ = {CHOSEN_DELTA} m)")
ax[1].grid(alpha=.3); ax[1].legend(loc="lower right")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "figures", "10_candidate_rule_choice.png"), dpi=130); plt.show()

## 11. Plots (all from the tables above)

In [ ]:
FIG = os.path.join(OUT_DIR, "figures")
PCOL = {"train": "#1f77b4", "student_test": "#ff7f0e"}

# 10.1 observations per trip
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for p in PARTITIONS:
    ax[0].hist(gps.query("partition==@p").n_points, bins=30, alpha=.7, label=p, color=PCOL[p])
ax[0].set(title="GPS observations per trip", xlabel="points", ylabel="trips"); ax[0].legend()
for p in PARTITIONS:
    ax[1].hist(gps.query("partition==@p").duration_min, bins=30, alpha=.7, label=p, color=PCOL[p])
ax[1].set(title="Trip duration", xlabel="minutes", ylabel="trips"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{FIG}/01_points_and_duration.png", dpi=130); plt.show()

# 10.2 sampling intervals and gaps
bins = [0, 1, 2, 5, 10, 60, np.inf]; labels = ["1 s", "2 s", "3–5 s", "6–10 s", "11–60 s", ">60 s"]
cats = pd.cut(all_dt.dt, bins=bins, labels=labels)
share = pd.crosstab(cats, all_dt.partition, normalize="columns") * 100
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
share.plot.bar(ax=ax[0], color=[PCOL[c] for c in share.columns], logy=True)
ax[0].set(title="Time between consecutive fixes (% of steps, log scale)", xlabel="", ylabel="% of steps")
g = gps.sort_values("max_gap_s", ascending=False)
ax[1].bar(range(len(g)), g.max_gap_s, color=[PCOL[p] for p in g.partition]); ax[1].set_yscale("log")
ax[1].axhline(LARGE_GAP_S, ls="--", c="k", lw=1); ax[1].set(title="Largest gap per trip (sorted)", xlabel="trip rank", ylabel="seconds (log)")
plt.tight_layout(); plt.savefig(f"{FIG}/02_sampling_gaps.png", dpi=130); plt.show()

# 10.3 speed distribution
fig, ax = plt.subplots(figsize=(7, 4))
for p in PARTITIONS:
    ax.hist(all_speed.query("partition==@p").speed.clip(0, 70), bins=70, alpha=.6, density=True, label=p, color=PCOL[p])
ax.set(title="Speed between consecutive 1-s fixes (clipped at 70 m/s)", xlabel="m/s", ylabel="density"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG}/03_speed.png", dpi=130); plt.show()

# 10.4 GPS-to-route distance
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for p in PARTITIONS:
    ax[0].hist(all_d2route.query("partition==@p").d2route.clip(0, 100), bins=100, alpha=.6, density=True, label=p, color=PCOL[p])
ax[0].set_yscale("log"); ax[0].set(title="Distance GPS point → own reference route (clipped 100 m)", xlabel="m", ylabel="density (log)"); ax[0].legend()
g = geo.sort_values("d2route_p90_m")
ax[1].bar(range(len(g)), g.d2route_p90_m, color=[PCOL[p] for p in g.partition])
ax[1].set(title="90th-percentile distance to route, per trip", xlabel="trip rank", ylabel="m")
plt.tight_layout(); plt.savefig(f"{FIG}/04_gps_to_route.png", dpi=130); plt.show()

# 10.5 network size & connectivity
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for p in PARTITIONS:
    q = net.query("partition==@p"); ax[0].scatter(q.n_nodes, q.n_arcs, label=p, color=PCOL[p], s=18)
ax[0].set(xscale="log", yscale="log", title="Road-network size per trip", xlabel="nodes", ylabel="directed arcs"); ax[0].legend()
g = net.sort_values("largest_scc_frac")
ax[1].bar(range(len(g)), g.largest_scc_frac, color=[PCOL[p] for p in g.partition])
ax[1].set(title="Share of nodes in largest strongly-connected component", xlabel="trip rank", ylabel="fraction", ylim=(0, 1))
plt.tight_layout(); plt.savefig(f"{FIG}/05_network_size_connectivity.png", dpi=130); plt.show()

# 10.6 world map of trip locations (static)
fig, ax = plt.subplots(figsize=(11, 5))
for p in PARTITIONS:
    q = gps.query("partition==@p"); ax.scatter(q.lon0, q.lat0, label=p, color=PCOL[p], s=25)
ax.set(xlim=(-180, 180), ylim=(-60, 80), title="Trip locations (mean GPS position)", xlabel="longitude", ylabel="latitude")
ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG}/06_trip_locations.png", dpi=130); plt.show()

# 10.7 candidate-set size vs radius
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
cands.boxplot(column="n_candidates", by="radius_m", ax=ax[0], showfliers=False)
ax[0].set(title="PRELIMINARY (1,000 pts/trip): arcs within radius", xlabel="radius (m)", ylabel="# arcs"); plt.suptitle("")
cov = cands.groupby(["radius_m", "partition"]).route_arc_within_r.mean().unstack()
cov.plot.bar(ax=ax[1], color=[PCOL[c] for c in cov.columns], ylim=(0.9, 1.0))
ax[1].legend(loc="lower right")
ax[1].set(title="PRELIMINARY, direction-blind coverage — final rule: section 10", xlabel="radius (m)", ylabel="fraction")
plt.tight_layout(); plt.savefig(f"{FIG}/07_candidates_vs_radius.png", dpi=130); plt.show()

# 10.8 gallery of trajectories (train) in metres
show = gps.query("partition=='train'").trip_id.values[:12]
fig, axs = plt.subplots(3, 4, figsize=(14, 10))
for ax, tid in zip(axs.ravel(), show):
    N = NET[("train", tid)]; d = TRIPS[("train", tid)]
    P = N["to_xy"](d["track"].lon, d["track"].lat)
    rt = d["route"].arc_id.values
    for a in rt:
        u, v = N["arcs"][a]; ax.plot(*N["nxy"][[u, v]].T, c="green", lw=2, alpha=.6)
    ax.plot(P[:, 0], P[:, 1], ".", ms=1.5, c="#1f77b4")
    ax.set_title(f"train trip {tid}", fontsize=9); ax.set_aspect("equal"); ax.tick_params(labelsize=6)
fig.suptitle("GPS points (blue) over reference route (green), local metres", y=1.0)
plt.tight_layout(); plt.savefig(f"{FIG}/08_trajectory_gallery.png", dpi=130); plt.show()

# 10.9 one trip close-up with road network (static preview of the Folium map)
def static_trip(part, tid, ax, buffer_m=150, zoom=None):
    N = NET[(part, tid)]; d = TRIPS[(part, tid)]; ar, nxy = N["arcs"], N["nxy"]
    P = N["to_xy"](d["track"].lon, d["track"].lat)
    tt = cKDTree(P)
    mid = (nxy[ar[:, 0]] + nxy[ar[:, 1]]) / 2
    near = (tt.query(mid)[0] < buffer_m) | (tt.query(nxy[ar[:, 0]])[0] < buffer_m)
    for u, v in ar[near]:
        ax.plot(*nxy[[u, v]].T, c="0.75", lw=.6)
    for a in d["route"].arc_id.values:
        u, v = ar[a]; ax.plot(*nxy[[u, v]].T, c="green", lw=2.2)
    ax.plot(P[:, 0], P[:, 1], ".", ms=2 if zoom is None else 6, c="#1f77b4", zorder=5)
    if zoom is not None:
        ax.set_xlim(zoom[0]); ax.set_ylim(zoom[1])
    ax.set_aspect("equal"); ax.set_title(f"{part} trip {tid}: network (grey), route (green), GPS (blue)", fontsize=9)

fig, ax = plt.subplots(1, 2, figsize=(14, 7))
static_trip("train", 1, ax[0])
P1 = NET[("train", 1)]["to_xy"](TRIPS[("train", 1)]["track"].lon, TRIPS[("train", 1)]["track"].lat)
c = P1[60]   # point 60 lies at a motorway interchange: separate parallel carriageways + ramps
static_trip("train", 1, ax[1], zoom=((c[0] - 300, c[0] + 300), (c[1] - 300, c[1] + 300)))
ax[1].set_title("zoom 600 m × 600 m around GPS point 60: interchange with parallel carriageways and ramps", fontsize=9)
plt.tight_layout(); plt.savefig(f"{FIG}/09_trip1_network_closeup.png", dpi=130); plt.show()

## 12. Interactive Folium maps
Layers: road network near the trajectory (grey), reference route (green), raw GPS points (blue),
time gaps > 10 s (red dashed), start/end markers. Layer control + legend included.
Saved as HTML in `eda_outputs/maps/` (open in a browser; screenshot for the report).

In [ ]:
import folium

def folium_trip_map(part, tid, buffer_m=150, max_points=4000):
    d = TRIPS[(part, tid)]; N = NET[(part, tid)]; ar = N["arcs"]
    tr, nd = d["track"], d["nodes"]
    m = folium.Map(location=[tr.lat.mean(), tr.lon.mean()], zoom_start=14, tiles="cartodbpositron", control_scale=True)
    # road network near the track (one line per undirected pair to keep the map light)
    P = N["to_xy"](tr.lon, tr.lat); tt = cKDTree(P); nxy = N["nxy"]
    mid = (nxy[ar[:, 0]] + nxy[ar[:, 1]]) / 2
    near = np.where(tt.query(mid)[0] < buffer_m)[0]
    seen, fg_net = set(), folium.FeatureGroup(name=f"Road network (<{buffer_m} m)")
    for a in near:
        u, v = ar[a]; k = (min(u, v), max(u, v))
        if k in seen: continue
        seen.add(k)
        folium.PolyLine([[nd.lat[u], nd.lon[u]], [nd.lat[v], nd.lon[v]]], color="#888888", weight=1.5, opacity=.6).add_to(fg_net)
    fg_net.add_to(m)
    # reference route (drawn arc by arc so breaks stay visible)
    fg_rt = folium.FeatureGroup(name="Reference route (ground truth)")
    for a in d["route"].arc_id.values:
        u, v = ar[a]
        folium.PolyLine([[nd.lat[u], nd.lon[u]], [nd.lat[v], nd.lon[v]]], color="green", weight=5, opacity=.7,
                        tooltip=f"route arc {a}").add_to(fg_rt)
    fg_rt.add_to(m)
    # GPS points
    fg_gps = folium.FeatureGroup(name="Raw GPS points")
    idx = np.linspace(0, len(tr) - 1, min(max_points, len(tr))).astype(int)
    for i in idx:
        folium.CircleMarker([tr.lat[i], tr.lon[i]], radius=2, color="#1f77b4", fill=True, fill_opacity=.8,
                            weight=0, popup=f"pt {i}, t={tr.t[i]:.0f}s").add_to(fg_gps)
    fg_gps.add_to(m)
    # gaps
    fg_gap = folium.FeatureGroup(name="Time gaps > 10 s")
    dt = np.diff(tr.t.values)
    for i in np.where(dt > LARGE_GAP_S)[0]:
        folium.PolyLine([[tr.lat[i], tr.lon[i]], [tr.lat[i + 1], tr.lon[i + 1]]], color="red", weight=3,
                        dash_array="6", tooltip=f"gap {dt[i]:.0f} s").add_to(fg_gap)
    fg_gap.add_to(m)
    folium.Marker([tr.lat.iloc[0], tr.lon.iloc[0]], tooltip="start", icon=folium.Icon(color="green", icon="play")).add_to(m)
    folium.Marker([tr.lat.iloc[-1], tr.lon.iloc[-1]], tooltip="end", icon=folium.Icon(color="red", icon="stop")).add_to(m)
    m.fit_bounds([[tr.lat.min(), tr.lon.min()], [tr.lat.max(), tr.lon.max()]])
    folium.LayerControl(collapsed=False).add_to(m)
    legend = f"""<div style="position:fixed;bottom:25px;left:25px;z-index:9999;background:white;padding:8px 10px;
      border:1px solid #999;border-radius:6px;font-size:13px">
      <b>{part} trip {tid}</b><br>
      <span style="color:#888">&#9473;&#9473;</span> road network<br>
      <span style="color:green">&#9473;&#9473;</span> reference route<br>
      <span style="color:#1f77b4">&#9679;</span> raw GPS<br>
      <span style="color:red">- - -</span> gap &gt; 10 s</div>"""
    m.get_root().html.add_child(folium.Element(legend))
    return m

# typical trip, 42-h gap, route breaks, parked off-network (54), route ends early (2), a student-test trip
MAP_TRIPS = [("train", 1), ("train", 90), ("train", 75), ("train", 54), ("train", 2), ("student_test", 0)]
for part, tid in MAP_TRIPS:
    mp = folium_trip_map(part, tid)
    mp.save(os.path.join(OUT_DIR, "maps", f"trip_{tid:08d}_{part}.html"))
    print("saved map", part, tid)

# overview map of all trips
ov = folium.Map(location=[30, 10], zoom_start=2, tiles="cartodbpositron")
for p in PARTITIONS:
    fg = folium.FeatureGroup(name=p)
    for r in gps.query("partition==@p").itertuples():
        folium.CircleMarker([r.lat0, r.lon0], radius=5, color=PCOL[p], fill=True, fill_opacity=.9,
                            popup=f"{p} trip {r.trip_id}: {r.n_points} pts, {r.path_km:.1f} km").add_to(fg)
    fg.add_to(ov)
folium.LayerControl(collapsed=False).add_to(ov)
ov.save(os.path.join(OUT_DIR, "maps", "overview_all_trips.html"))
folium_trip_map("train", 1)   # display one map inline in the notebook

## 13. Validity cross-checks — not trusting the files blindly
The dataset guide says to **verify** column meanings, coordinate order and plausibility, and not to treat the
reference route as error-free. Sections 2–9 checked structure and values. Here we test the *assumptions*:

| # | Assumption we were relying on | Test |
|---|---|---|
| V1 | Column 1 = longitude, column 2 = latitude | latitude must lie in ±90; any trip with \|col1\| > 90 proves col1 is longitude; centroids land on real cities |
| V2 | Timestamps are in seconds | integer-valued? implied speeds plausible for road vehicles? |
| V3 | GPS has no frozen/duplicate fixes or single-point spikes | identical consecutive coordinates; "jump-and-return" spikes |
| V4 | Nodes are unique and the network covers the trajectory | duplicate node coordinates; GPS points outside the network extent |
| V5 | The route is driven in arc direction and in time order | GPS heading vs bearing of the nearest route arc; does nearest-route-arc index move forward in time? |
| V6 | Route doesn't use broken arcs | route arcs that are self-loops, zero-length or duplicates |
| V7 | Trips are independent (no duplicated trips, no train/test leakage) | file hashes; geometric overlap between trips in the same area |

In [ ]:
import hashlib

# ---- V1: coordinate order ----------------------------------------------------------------
v1 = []
for (part, tid), d in TRIPS.items():
    tr, nd = d["track"], d["nodes"]
    v1.append({"partition": part, "trip_id": tid,
               "track_col2_within_90": bool((tr.lat.abs() <= 90).all()),
               "nodes_col2_within_90": bool((nd.lat.abs() <= 90).all()),
               "col1_exceeds_90_somewhere": bool((tr.lon.abs() > 90).any() or (nd.lon.abs() > 90).any()),
               "lon0": round(tr.lon.mean(), 3), "lat0": round(tr.lat.mean(), 3)})
v1 = pd.DataFrame(v1)
print("V1  column 2 within ±90 in every track and node file:",
      bool(v1.track_col2_within_90.all() and v1.nodes_col2_within_90.all()))
print("V1  trips where |column 1| > 90 (so column 1 can only be longitude):",
      v1[v1.col1_exceeds_90_somewhere].trip_id.tolist())

# ---- V2: timestamps -----------------------------------------------------------------------
v2 = []
for (part, tid), d in TRIPS.items():
    t = d["track"].t.values
    v2.append({"partition": part, "trip_id": tid, "all_integer": bool(np.all(t == np.round(t))),
               "t_first": t[0]})
v2 = pd.DataFrame(v2)
print("\nV2  trips with non-integer timestamps:", v2[~v2.all_integer].trip_id.tolist(),
      "-> fractional parts are tiny constant offsets / clock drift (inspect below); units are still seconds")
for tid in v2[~v2.all_integer].trip_id:
    part = v2.set_index("trip_id").loc[tid, "partition"]
    t = TRIPS[(part, tid)]["track"].t.values; f = t - np.round(t)
    print(f"    trip {tid}: {np.mean(f != 0):.1%} non-integer, distinct fractional parts e.g. {np.unique(np.round(f[f != 0], 3))[:4]}")
print("V2  median 1-s speed over all trips: %.1f m/s (%.0f km/h) — plausible for road traffic if t is in seconds"
      % (all_speed.speed.median(), all_speed.speed.median() * 3.6))

# ---- V3: frozen fixes and spikes -----------------------------------------------------------
v3 = []
for (part, tid), d in TRIPS.items():
    tr = d["track"]
    xy = NET[(part, tid)]["to_xy"](tr.lon, tr.lat)
    same = (np.diff(tr.lon.values) == 0) & (np.diff(tr.lat.values) == 0)
    dup_rows = int(tr.duplicated().sum())
    # spike: point i far from both neighbours (>50 m) while neighbours are close to each other (<0.3 × that)
    a = np.linalg.norm(xy[1:-1] - xy[:-2], axis=1); b = np.linalg.norm(xy[2:] - xy[1:-1], axis=1)
    c = np.linalg.norm(xy[2:] - xy[:-2], axis=1)
    spikes = int(((a > 50) & (b > 50) & (c < 0.3 * np.minimum(a, b))).sum())
    v3.append({"partition": part, "trip_id": tid, "n_identical_consecutive_fixes": int(same.sum()),
               "n_duplicate_rows": dup_rows, "n_spikes": spikes})
v3 = pd.DataFrame(v3)
print("\nV3  identical consecutive fixes:", v3.n_identical_consecutive_fixes.sum(),
      "| fully duplicated rows:", v3.n_duplicate_rows.sum(),
      "| jump-and-return spikes:", v3.n_spikes.sum(), "in trips", v3[v3.n_spikes > 0].trip_id.tolist())

# ---- V4: nodes & coverage ------------------------------------------------------------------
v4 = []
for (part, tid), d in TRIPS.items():
    nd = d["nodes"]; N = NET[(part, tid)]
    P = N["to_xy"](d["track"].lon, d["track"].lat)
    lo, hi = N["nxy"].min(0), N["nxy"].max(0)
    outside = ~((P >= lo) & (P <= hi)).all(1)
    v4.append({"partition": part, "trip_id": tid,
               "dup_node_coords": int(nd.duplicated().sum()),
               "zero_len_arcs": int((N["len"] < 0.01).sum()),
               "gps_pts_outside_network_bbox": int(outside.sum()),
               "net_margin_min_m": float(min((P.min(0) - lo).min(), (hi - P.max(0)).min()))})
v4 = pd.DataFrame(v4)
print("\nV4  nodes sharing identical coordinates:", v4.dup_node_coords.sum(), "in", (v4.dup_node_coords > 0).sum(), "trips",
      "| zero-length arcs:", v4.zero_len_arcs.sum(),
      "| GPS points outside network extent:", v4.gps_pts_outside_network_bbox.sum(),
      "| smallest margin between GPS extent and network edge: %.0f m" % v4.net_margin_min_m.min())

# ---- V5: route direction and order ---------------------------------------------------------
def bearing_deg(v):
    return np.degrees(np.arctan2(v[:, 0], v[:, 1])) % 360       # 0 = north, clockwise

v5 = []
for (part, tid), d in TRIPS.items():
    N = NET[(part, tid)]; ar, nxy = N["arcs"], N["nxy"]
    tr = d["track"]; P = N["to_xy"](tr.lon, tr.lat)
    rt = d["route"].arc_id.values
    A, B = nxy[ar[rt, 0]], nxy[ar[rt, 1]]
    AB = B - A; L2 = np.maximum((AB ** 2).sum(1), 1e-12); ab = bearing_deg(AB)
    # GPS heading from positions 4 samples apart (smooths jitter); only where moving > 3 m/s
    h = np.full(len(P), np.nan)
    if len(P) > 4:
        v = P[4:] - P[:-4]; sp = np.linalg.norm(v, axis=1) / np.maximum(tr.t.values[4:] - tr.t.values[:-4], 1)
        hh = bearing_deg(v); hh[sp < 3] = np.nan; h[2:-2] = hh
    moving = ~np.isnan(h)
    d_any = np.empty(len(P)); d_same = np.full(len(P), np.inf); k_any = np.empty(len(P), int); k_same = np.empty(len(P), int)
    for s0 in range(0, len(P), 1500):
        sl = slice(s0, s0 + 1500)
        AP = P[sl, None] - A[None]
        t = np.clip((AP * AB[None]).sum(2) / L2[None], 0, 1)
        D = np.linalg.norm(AP - t[..., None] * AB[None], axis=2)
        k_any[sl] = D.argmin(1); d_any[sl] = D.min(1)
        diff = np.abs((h[sl, None] - ab[None] + 180) % 360 - 180)
        Ds = np.where(diff < 90, D, np.inf)                       # only route arcs pointing the way we move
        k_same[sl] = Ds.argmin(1); d_same[sl] = Ds.min(1)
    near = moving & (d_any < 20)
    # naive (direction-blind) test: is the nearest route arc pointing the opposite way?
    diff_naive = np.abs((h[near] - ab[k_any[near]] + 180) % 360 - 180)
    # heading-aware test: is there a SAME-direction route arc within 20 m?
    same_ok = d_same[near] < 20
    kk = k_same[moving & (d_same < 20)]
    rho = pd.Series(np.arange(len(kk))).corr(pd.Series(kk), method="spearman") if len(kk) > 2 else np.nan
    rset = set(map(tuple, ar[rt]))
    # a genuinely reversed route segment would show as a LONG run of consecutive points without a same-direction arc;
    # isolated misses happen at sharp turns / slow speed where GPS heading is noisy
    bad_idx = np.where(near & (d_same >= 20))[0]
    runs = np.split(bad_idx, np.where(np.diff(bad_idx) > 1)[0] + 1) if len(bad_idx) else []
    v5.append({"max_run_without_same_dir_arc": max((len(r) for r in runs), default=0),"partition": part, "trip_id": tid, "n_moving_near_route": int(near.sum()),
               "route_arcs_with_reverse_also_in_route": int(sum((b_, a_) in rset for a_, b_ in ar[rt])),
               "naive_frac_nearest_route_arc_opposite": float((diff_naive > 90).mean()) if near.any() else np.nan,
               "frac_with_same_direction_route_arc_20m": float(same_ok.mean()) if near.any() else np.nan,
               "spearman_time_vs_route_index": rho, "n_backward_jumps": int((np.diff(kk) < -2).sum())})
v5 = pd.DataFrame(v5)
print("\nV5  Direction-blind test (misleading): share of moving on-route points whose nearest route arc points the"
      " opposite way: mean %.3f, worst %.3f" % (v5.naive_frac_nearest_route_arc_opposite.mean(),
                                               v5.naive_frac_nearest_route_arc_opposite.max()))
print("V5  Why: out-and-back trips — route contains both u→v and v→u; worst trip:",
      v5.loc[v5.route_arcs_with_reverse_also_in_route.idxmax(), ["trip_id", "route_arcs_with_reverse_also_in_route"]].tolist())
print("V5  Heading-aware test: share of moving on-route points that have a SAME-direction route arc within 20 m: "
      "min %.3f, median %.3f" % (v5.frac_with_same_direction_route_arc_20m.min(), v5.frac_with_same_direction_route_arc_20m.median()))
print("V5  Heading-aware Spearman(time, route index): min %.3f, median %.3f" % (
      v5.spearman_time_vs_route_index.min(), v5.spearman_time_vs_route_index.median()))
SUSTAINED_RUN = 20   # >=20 consecutive points (~20 s) = sustained mismatch, worth inspecting by eye
print("V5  Trips with a sustained run (>= %d consecutive points) lacking a same-direction route arc:" % SUSTAINED_RUN,
      v5[v5.max_run_without_same_dir_arc >= SUSTAINED_RUN][["trip_id", "max_run_without_same_dir_arc"]].values.tolist(),
      "| all other misses are short runs (<%d pts) at sharp turns / slow speed where GPS heading is noisy" % SUSTAINED_RUN)
display(v5.sort_values("spearman_time_vs_route_index").head(5).round(3))

# ---- V6: route arcs that are self-loops / zero-length / duplicates / revisited ------------
v6 = []
for (part, tid), d in TRIPS.items():
    N = NET[(part, tid)]; ar = N["arcs"]; rt = d["route"].arc_id.values
    pair = pd.Series(list(map(tuple, ar)))
    dup_mask = pair.duplicated(keep=False).values
    v6.append({"partition": part, "trip_id": tid,
               "route_selfloops": int((ar[rt, 0] == ar[rt, 1]).sum()),
               "route_zero_len": int((N["len"][rt] < 0.01).sum()),
               "route_uses_duplicate_arc": int(dup_mask[rt].sum()),
               "route_nonconsecutive_revisits": int(len(rt) - len(np.unique(rt)))})
v6 = pd.DataFrame(v6)
print("\nV6  route arcs that are self-loops:", v6.route_selfloops.sum(), "| zero-length:", v6.route_zero_len.sum(),
      "| have a duplicate twin arc:", v6.route_uses_duplicate_arc.sum(), "in trips", v6[v6.route_uses_duplicate_arc > 0].trip_id.tolist(),
      "| arcs revisited later in the route:", v6.route_nonconsecutive_revisits.sum(), "in trips",
      v6[v6.route_nonconsecutive_revisits > 0].trip_id.tolist())

# ---- V7: duplicated trips / leakage between partitions -------------------------------------
hashes = {}
for r in inventory.itertuples():
    base = os.path.join(DATA_ROOT, r.partition, f"{r.trip_id:08d}", f"{r.trip_id:08d}")
    hashes[(r.partition, r.trip_id)] = {e: hashlib.md5(open(f"{base}.{e}", "rb").read()).hexdigest() for e in EXT}
for e in EXT:
    hs = pd.Series({k: v[e] for k, v in hashes.items()})
    dup = hs[hs.duplicated(keep=False)]
    print(f"V7  identical .{e} files across trips:", "none" if dup.empty else dup.groupby(dup).apply(lambda s: list(s.index)).tolist())

keys = list(TRIPS.keys())
cent = np.array([[TRIPS[k]["track"].lon.mean(), TRIPS[k]["track"].lat.mean()] for k in keys])
overlap_rows = []
for i in range(len(keys)):
    for j in range(len(keys)):
        if i == j or haversine(*cent[i], *cent[j]) > 50_000:
            continue
        to_xy = make_projection(*cent[i])
        Pi = to_xy(TRIPS[keys[i]]["track"].lon, TRIPS[keys[i]]["track"].lat)
        Pj = to_xy(TRIPS[keys[j]]["track"].lon, TRIPS[keys[j]]["track"].lat)
        frac = float((cKDTree(Pj).query(Pi)[0] < 15).mean())
        overlap_rows.append({"trip_a": f"{keys[i][0]}:{keys[i][1]}", "trip_b": f"{keys[j][0]}:{keys[j][1]}",
                             "cross_partition": keys[i][0] != keys[j][0],
                             "frac_a_points_within_15m_of_b": round(frac, 3)})
overlap = pd.DataFrame(overlap_rows).sort_values("frac_a_points_within_15m_of_b", ascending=False)
overlap.to_csv(os.path.join(OUT_DIR, "tables", "10_trip_overlap.csv"), index=False)
print("\nV7  trip pairs within 50 km of each other:", len(overlap) // 2,
      "| pairs where >50% of A's points lie within 15 m of B's track:")
display(overlap[overlap.frac_a_points_within_15m_of_b > 0.5])

validity = (v1.merge(v2, on=["partition", "trip_id"]).merge(v3, on=["partition", "trip_id"])
            .merge(v4, on=["partition", "trip_id"]).merge(v5, on=["partition", "trip_id"])
            .merge(v6, on=["partition", "trip_id"]))
validity.to_csv(os.path.join(OUT_DIR, "tables", "09_validity_checks.csv"), index=False)

## 14. Data-quality log (auto-generated from the tables above)

In [ ]:
S = gps.merge(net, on=["partition", "trip_id"]).merge(routes, on=["partition", "trip_id"]) \
       .merge(geo, on=["partition", "trip_id"]).merge(nearest, on=["partition", "trip_id"])
S.to_csv(os.path.join(OUT_DIR, "tables", "00_eda_summary_per_trip.csv"), index=False)

def ids(q):
    return ", ".join(f"{p[:5]}:{t}" for p, t in S.query(q)[["partition", "trip_id"]].values) or "none"

cov = cands.groupby("radius_m").route_arc_within_r.mean()
_off = set(S.query("frac_off_network_gt50m>0.05").trip_id)
cov_clean = cands[~cands.trip_id.isin(_off)].groupby("radius_m").route_arc_within_r.mean()
log = f"""# Data-quality log (generated by 01_eda)
Partitions analysed: train ({(S.partition=='train').sum()} trips), student_test ({(S.partition=='student_test').sum()} trips).

## GPS (.track)
- Missing rows: {S.n_missing_rows.sum()}; out-of-range coordinates: {S.n_out_of_range.sum()}; duplicate timestamps: {S.n_dup_timestamps.sum()}; all timestamps strictly increasing: {S.time_monotonic.all()}.
- Share of 1-s steps (all trips): {(all_dt.dt==1).mean():.3%}.
- Trips with a gap > 60 s: {ids('n_gaps_gt60s>0')}.
- Trips with a gap > 1 h: {ids('max_gap_s>3600')}.
- Trips with at least one 1-s step faster than {SPEED_JUMP_MS} m/s (216 km/h): {ids('n_speed_jumps>0')}.
- Trips whose 99th-percentile speed exceeds {SPEED_JUMP_MS} m/s (sustained, not a single glitch — fast highway/rail-like or timestamp issue): {ids(f'speed_p99_ms>{SPEED_JUMP_MS}')}.
- Trips parked/dwelling (60-s average speed <0.5 m/s) for >30% of the time: {ids('frac_dwell_60s>0.3')}.

## Road network (.nodes/.arcs)
- Arcs referencing non-existent nodes: {S.arcs_bad_node_ref.sum()}.
- Self-loop arcs: {S.self_loops.sum()} (trips: {ids('self_loops>0')}).
- Duplicate arcs: {S.duplicate_arcs.sum()} (in {(S.duplicate_arcs>0).sum()} trips).
- Isolated nodes (no arcs): {S.isolated_nodes.sum()} (in {(S.isolated_nodes>0).sum()} trips).
- Median share of two-way arcs: {S.frac_two_way_arcs.median():.3f}.
- Weakly connected components per trip: median {S.n_weak_components.median():.0f}, max {S.n_weak_components.max()}.
- Largest strongly-connected component share: min {S.largest_scc_frac.min():.3f}, median {S.largest_scc_frac.median():.3f}.

## Reference route (.route)
- Invalid arc ids: {S.n_invalid_ids.sum()}.
- Routes with breaks (consecutive arcs not connected): {ids('n_breaks>0')}.
- Consecutive repeated arcs: {S.n_consecutive_repeats.sum()}.
- Route length / GPS path length: median {S.route_to_gps_len_ratio.median():.3f}, range {S.route_to_gps_len_ratio.min():.3f}–{S.route_to_gps_len_ratio.max():.3f}.
  Raw 1-Hz GPS path length is inflated by jitter (esp. when slow/parked). Using 10-s subsampled path instead: route/GPS ratio median {(S.route_km/S.path_km_10s_subsampled).median():.3f}.
- Reference route ends well before the GPS track (last point >100 m from route): {ids('last_pt_d2route_m>100')}; starts late (first point >100 m): {ids('first_pt_d2route_m>100')}.
- GPS points per route arc: median {S.pts_per_route_arc.median():.2f} → labels must be derived per point.

## GPS vs route / network
- Distance to own route: median {all_d2route.d2route.median():.2f} m, p95 {all_d2route.d2route.quantile(.95):.2f} m, p99 {all_d2route.d2route.quantile(.99):.2f} m.
- Trips with >5% of points more than 50 m from route: {ids('frac_gt50m>0.05')}.
- Trips with >5% of points more than 50 m from ANY road arc (off-network, e.g. parked inside a site): {ids('frac_off_network_gt50m>0.05')}.
- Nearest arc is a route arc (directed): mean {S.frac_nearest_on_route_directed.mean():.3f}; (undirected): mean {S.frac_nearest_on_route_undirected.mean():.3f}.
- Projection check: worst p99 relative error on consecutive points {proj_check['consec_p99_rel_err_%'].max():.4f}%, worst far-pair error {proj_check['far_max_rel_err_%'].max():.3f}%.
- (Preliminary, section 9: 1,000 pts/trip, direction-blind) share of points with a route arc within 25/50/100 m: {cov.get(25, float('nan')):.4f} / {cov.get(50, float('nan')):.4f} / {cov.get(100, float('nan')):.4f}.
  Superseded by section 10.

## Candidate rule (section 10 — selected on train only)
- Robust GPS noise σ = {sigma:.2f} m; d* p95 {mv.dstar.quantile(.95):.1f} m vs Gaussian-model p95 {1.96*sigma:.1f} m → heavy tail.
- Fixed 50 m (train-clean): coverage {cov_f(rc_clean, 50):.4f}, mean K {rc_clean['K50'].mean():.2f}, empty sets {(rc_clean['K50']==0).mean():.4f}.
- CHOSEN adaptive rule: arcs within min(d_nearest + {CHOSEN_DELTA} m, {RC_RMAX:.0f} m): coverage train-clean {RC_CHOSEN.coverage_train_clean:.4f}, train-all {RC_CHOSEN.coverage_train_all:.4f}, student-test (check) {RC_CHOSEN.coverage_student_test:.4f}; mean K {RC_CHOSEN.mean_K_clean:.2f}; empty sets {RC_CHOSEN.empty_sets_clean:.4f}.
- Same-budget fixed radius r={r_eq} m reaches only {cov_f(rc_clean, r_eq):.4f}.
"""
log += f"""
## Validity cross-checks (section 13)
- V1 coordinate order: column 2 always within ±90; column 1 exceeds ±90 in trips {v1[v1.col1_exceeds_90_somewhere].trip_id.tolist()} → column 1 = longitude, confirmed. Centroids fall on real cities (overview map).
- V2 timestamps: non-integer in trips {v2[~v2.all_integer].trip_id.tolist()} (constant sub-second offsets / ~0.016 s drift); units are seconds (median speed {all_speed.speed.median():.1f} m/s). Use tolerances, never `dt == 1`.
- V3 GPS: {v3.n_identical_consecutive_fixes.sum()} identical consecutive fixes, {v3.n_duplicate_rows.sum()} duplicate rows, {v3.n_spikes.sum()} jump-and-return spikes (trips {v3[v3.n_spikes > 0].trip_id.tolist()}).
- V4 network: {v4.dup_node_coords.sum()} nodes share coordinates with another node ({(v4.dup_node_coords > 0).sum()} trips); {v4.zero_len_arcs.sum()} zero-length arcs; 0 GPS points outside network extent (min margin {v4.net_margin_min_m.min():.0f} m).
- V5 route direction/order: direction-blind check looks bad (worst {v5.naive_frac_nearest_route_arc_opposite.max():.2f}) only because of out-and-back trips; heading-aware: ≥{v5.frac_with_same_direction_route_arc_20m.min():.3f} of moving on-route points have a same-direction route arc within 20 m; sustained runs (>= {SUSTAINED_RUN} pts) without one: {v5[v5.max_run_without_same_dir_arc >= SUSTAINED_RUN][['trip_id', 'max_run_without_same_dir_arc']].values.tolist()} (trip 66: ~65 s driving opposite to the listed route arcs while a same-direction arc exists ~5 m away = local reference error; trip 54: parked jitter); all other misses are short runs at sharp turns / slow speed; Spearman(time, route index) median {v5.spearman_time_vs_route_index.median():.3f}, min {v5.spearman_time_vs_route_index.min():.3f} (trip {v5.loc[v5.spearman_time_vs_route_index.idxmin(), 'trip_id']}).
  → Label derivation in M3 MUST be heading-aware and order-aware.
- V6 route arcs: self-loops {v6.route_selfloops.sum()}, zero-length {v6.route_zero_len.sum()}, with a duplicate twin {v6.route_uses_duplicate_arc.sum()} (trips {v6[v6.route_uses_duplicate_arc > 0].trip_id.tolist()}), revisited later {v6.route_nonconsecutive_revisits.sum()} (trips {v6[v6.route_nonconsecutive_revisits > 0].trip_id.tolist()}).
- V7 independence: no identical files across trips. Strong geometric overlap (>50% of points within 15 m): {overlap[overlap.frac_a_points_within_15m_of_b > 0.5][['trip_a', 'trip_b']].values.tolist()}; cross-partition pairs above 50%: {int(overlap[(overlap.frac_a_points_within_15m_of_b > 0.5) & overlap.cross_partition].shape[0])}.
"""
with open(os.path.join(OUT_DIR, "data_quality_log.md"), "w") as f:
    f.write(log)
print(log)